## HyperDrone Renderer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rl-tools/rl-tools/blob/main/interface/python/hyperdrone/hyperdrone/examples/Renderer.ipynb)

This notebook shows how to use the renderer and the dynamics separately such that you can plug in your own logic.


### Install

Don't be frightened by the following install procedure, it just makes sure this notebook works on Google Colab as well as when run from the repo. For normal usage `pip install hyperdrone[examples]` is completely sufficient and does not require other setup or dependencies (apart from CUDA)

In [ ]:
import importlib.util
import site
import sysconfig
from pathlib import Path

checkout = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "interface/python/hyperdrone/pyproject.toml").is_file()), None)
if checkout is not None:
    %pip install -q -e "{checkout}/interface/python/hyperdrone[examples]"
    site.addsitedir(sysconfig.get_paths()["purelib"])  # activates the editable install's .pth without a kernel restart
elif importlib.util.find_spec("hyperdrone") is None:  # fresh runtime (e.g. Colab)
    %pip install -q "hyperdrone[examples]"
%pip install -q foundation-policy==1.0.1

We set up three main components here: the renderer, the dynamics simulation and a policy to give control commands (RAPTOR):

In [ ]:
import math
from pathlib import Path

import conta
import imageio.v2 as imageio
import numpy as np
from foundation_policy import Raptor

from hyperdrone import dynamics, render
from hyperdrone.camera import CameraSpec, RGB, Depth, Segmentation, Normals, Flow, FlowStatus
from hyperdrone.env import Rotorcraft
from hyperdrone.examples.data import RIGGED_MODELS, procthor_scene_path

WIDTH, HEIGHT = 512, 512
STEPS, FPS = 180, 30
MOTION_BLUR_SAMPLES = 8
TARGET = np.array([-3.92, -5.67, 1.0], dtype=np.float32)
START = TARGET + np.array([.5, 0, 0], dtype=np.float32)
CAMERA_PITCH = .3
MOUNT = render.make_transform(position=(.1, 0, .32), orientation_wxyz=(math.cos(CAMERA_PITCH / 2), 0, math.sin(CAMERA_PITCH / 2), 0))

sim = dynamics.Sim(num_drones=1, parameters={"dynamics": {"dynamics": {"model": "x500"}}}, device="auto")
sim.reset()
sim.state["position"] = START[None]
policy = Raptor()
policy.reset()
previous_action = np.zeros((1, sim.action_dim), dtype=np.float32)

scene = render.load_scene(procthor_scene_path(), fidelity="high")
drone = render.load_assembly(conta.resolve(RIGGED_MODELS["x500"]), fidelity="high")
rotors = sim.parameters["dynamics"]["dynamics"]
rig = Rotorcraft(drone, rotor_positions=rotors["rotor_positions"], rotor_torque_directions=rotors["rotor_torque_directions"])
assets = render.AssetPool()
drone_asset = assets.add_assembly(drone)

renderer = render.Renderer(
    width=WIDTH, height=HEIGHT, batch_size=2, cameras=(CameraSpec("front", (RGB("rgba8"),)),), fidelity="high",
    num_overlays=1, max_overlay_instances=drone.num_parts, max_overlays_per_camera=1,
    motion_blur_samples=MOTION_BLUR_SAMPLES, dynamic_motion_blur=True,
)
renderer.init(scene, asset_pool=assets)
renderer.attach(0, 0)  # the onboard camera (0) sees the drone's own overlay (0)
renderer.attach(1, 0)  # and so does the external camera (1)
placement = renderer.spawn(0, drone_asset, render.make_transform(position=START))
external = renderer.camera(position=TARGET + np.array([-1.2, -1.2, .7]), look_at=TARGET, fov=55).reshape(1, 12)


The policy is driving the dynamics simulation and the states from the dynamics simulation (pose of the drone & motor command states) are forwarded to the renderer. 

In [ ]:
def body_pose():
    return render.make_transform(position=sim.state["position"][0], orientation_wxyz=sim.state["orientation"][0])
def cameras():
    return np.concatenate([sim.camera_bases(mount=MOUNT, fov=100, aspect=renderer.aspect), external])

body_open, cameras_open, phase_open = body_pose(), cameras(), rig.default_coordinates
frames = []
for _ in range(STEPS):
    observation = sim.observe()
    observation[:, :3] -= TARGET  # Position error for the step response.
    action = policy.evaluate_step(np.concatenate([observation, previous_action], axis=1))
    sim.step(action)
    previous_action = action

    body_close, cameras_close = body_pose(), cameras()
    phase_close = phase_open + sim.rotor_rpm()[0] * (2 * np.pi / 60) * sim.dt
    renderer.set_transform_pair(placement, body_open, body_close, rig=rig, coordinates_open=phase_open, coordinates_close=phase_close)
    renderer.set_motion_blur_cameras(cameras_open, cameras_close)
    renderer.update()
    renderer.render()
    views = renderer.outputs[RGB("rgba8")].read()[..., :3]
    frames.append(np.concatenate([views[0], views[1]], axis=1))
    body_open, cameras_open, phase_open = body_close, cameras_close, np.remainder(phase_close, 2 * np.pi)

mp4_path = Path("hyperdrone_sim.mp4")
imageio.mimsave(mp4_path, frames, fps=FPS)
print(f"Saved {mp4_path} (Raptor onboard/self-occlusion | fixed external view | articulated propellers)")

from IPython.display import Video, display
display(Video(str(mp4_path), embed=True))

## Depth, segmentation, normals and optical flow

Outputs are composable. Flow is current-to-previous pixel displacement; `advance_flow()` snapshots camera and object poses before the next update. The first render seeds zero motion. These geometric outputs use the shutter-close pose, independently of motion blur.


In [ ]:
sensors = render.Renderer(width=128, height=128, batch_size=2, fidelity="low",
                          cameras=(CameraSpec("front", (RGB("rgba8"), Depth(), Segmentation(), Normals(), Flow(), FlowStatus())),))
sensor_scene = render.load_scene(procthor_scene_path(), fidelity="low", preserve_instances=True)
sensors.init(sensor_scene)
previous_cameras = cameras().copy()
sensors.set_cameras(previous_cameras)
sensors.render()
sensors.advance_flow()
current_cameras = previous_cameras.reshape(2, 4, 3).copy()
current_cameras[:, 0, 1] += 0.05
sensors.set_cameras(current_cameras)
sensors.render()
depth, labels = sensors.outputs[Depth()].read()[0], sensors.outputs[Segmentation()].read()[0]
normals, flow, status = sensors.outputs[Normals()].read()[0], sensors.outputs[Flow()].read()[0], sensors.outputs[FlowStatus()].read()[0]

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(depth, cmap="viridis")
axes[1].imshow(np.ma.masked_equal(labels, 0xFFFFFFFF), cmap="tab20", interpolation="nearest")
axes[2].imshow(np.clip(normals * 0.5 + 0.5, 0, 1))
axes[3].imshow(np.ma.masked_where(status != render.FlowStatus.VALID, np.linalg.norm(flow, axis=-1)), cmap="magma")
for axis, title in zip(axes, ("Depth (m)", "Instance IDs", "World normals", "Valid flow magnitude (px)")):
    axis.set_title(title)
    axis.axis("off")
plt.show()
